# Deterministic Modeling of Surgical Planning

This model assumes that the duration of each surgery is known and fixed (no uncertainty). The objective is to minimize the fixed costs of using the operating rooms and the variable costs related to overtime, while respecting a set of operational constraints.

## 1. Sets and Parameters

* $I = \{1, ..., n\}$: set of patients.
* $J$: set of surgical specialties.
* $K = \{1, ..., 7\}$: set of days of the week (1 = Monday, 6 = Saturday, 7 = Sunday).
* $s_i \in J$: specialty required for patient $i \in I$.
* $t_{i,fixe}$: minimum deterministic duration of the surgery for patient $i$.
* $C_{jk}$: fixed cost of using the room for specialty $j$ on day $k$.
* $C_{jk}^{\prime}$: hourly overtime cost for specialty $j$ on day $k$.
* $H = 8$: standard number of working hours per day.
* $H_{max\_sup} = 2$: maximum allowed overtime hours per day.

*(Note: The effective costs $C_{jk}$ and $C_{jk}^{\prime}$ are multiplied by 1.25 on Saturday and by 1.8 on Sunday compared to the weekday base cost).*

## 2. Decision Variables

* $x_{ik} \in \{0, 1\}$: takes the value 1 if patient $i$ is operated on day $k$, 0 otherwise.
* $y_{jk} \in \{0, 1\}$: takes the value 1 if the room is allocated to specialty $j$ on day $k$, 0 otherwise.
* $\tau_{jk} \ge 0$: number of overtime hours required for specialty $j$ on day $k$.

## 3. Objective Function

The objective is to minimize the sum of the fixed costs of opening the rooms and the costs related to overtime.

$$\min Z = \sum_{j \in J}\sum_{k \in K}C_{jk}y_{jk} + \sum_{j \in J}\sum_{k \in K}C_{jk}^{\prime}\tau_{jk}$$

## 4. Constraints

**1. Unique scheduling for each patient:**
Each patient $i$ must be operated on exactly once during the week.
$$\sum_{k \in K}x_{ik} = 1 \quad \forall i \in I$$

**2. Daily room capacity:**
The operating room can only accommodate a single specialty per day $k$.
$$\sum_{j \in J}y_{jk} \le 1 \quad \forall k \in K$$

**3. Specialty consistency:**
A patient $i$ can only be operated on day $k$ if the room is prepared for their specialty $s_i$ on that day.
$$x_{ik} \le y_{s_i, k} \quad \forall i \in I, \forall k \in K$$

**4. Definition and calculation of overtime:**
If the total time of scheduled surgeries for specialty $j$ on day $k$ exceeds the standard 8 hours, the difference defines the overtime $\tau_{jk}$.
$$\sum_{i \in I | s_i = j}t_{i,fixe}x_{ik} - H y_{jk} \le \tau_{jk} \quad \forall j \in J, \forall k \in K$$

**5. Legal overtime limit:**
Overtime cannot exceed the maximum allowed limit (2 hours). Furthermore, it can only occur if the room is open for this specialty.
$$0 \le \tau_{jk} \le H_{max\_sup} y_{jk} \quad \forall j \in J, \forall k \in K$$

**6. Nature of variables:**
Integrity constraints for binary variables.
$$x_{ik}, y_{jk} \in \{0, 1\}$$

In [1]:
import pulp
import pandas as pd


# 1. Data loading

The data is loaded from an Excel file named `surgery_data.xlsx` which contain the **patient ID**, **required specialty** (name and ID), **expected base duration** of the surgery and **maximum potential overtime**.

In [ ]:
# Excel file name
FILENAME = "data/surgery_data.xlsx"

try:
    # Load the first sheet (patients)
    df = pd.read_excel(FILENAME, sheet_name="Patients")
    df.columns = ['Patient_ID', 'Spec_ID', 'Specialty', 'Duration', 'Overtime']

    # Load the second sheet (costs)
    df_costs = pd.read_excel(FILENAME, sheet_name="Specialties")

except FileNotFoundError:
    print(f"ERROR: The file '{FILENAME}' was not found.")
    exit()

Let **$\alpha$** be a prudence factor for uncertainty management :
* **$\alpha$** $ \in  [0,1]$
* $0=$ optimistic : no overtime
* $1=$ pessimistic : full overtime

In [3]:
ALPHA_PRUDENCE = 0.5

**1. Data Harmonization and Sets Definition**

First, we normalize the input data to ensure consistency (removing spaces and converting to uppercase). Then, we define the main sets for the optimization problem: patients, surgical specialties, and the planning horizon (7 days).

In [4]:
# --- Name harmonization to avoid inconsistencies ---
df['Specialty'] = df['Specialty'].str.strip().str.upper()
df_costs['names'] = df_costs['names'].str.strip().str.upper()

# Sets definition
patients = [f"P{i}" for i in df['Patient_ID']]
specialties = df['Specialty'].unique().tolist()
days = list(range(1, 8))  # 1=Monday, ..., 7=Sunday
day_map = {
    1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 
    5: "Friday", 6: "Saturday", 7: "Sunday"
}

**2. Cost Extraction and Validation**

We extract the base costs for each specialty from the "Specialties" sheet. We also perform a consistency check to ensure that every patient's specialty has a corresponding cost defined in the system.

In [5]:
# --- 2. Cost extraction from the "Specialties" sheet ---
# Expected columns: "names", "fixed cost", "overtime cost (per h)"
base_fixed_costs = {}
base_overtime_costs = {}

for _, row in df_costs.iterrows():
    specialty_name = row['names']
    base_fixed_costs[specialty_name] = row['fixed cost']
    base_overtime_costs[specialty_name] = row['overtime cost (per h)']

# Consistency check
missing_specialties = [s for s in specialties if s not in base_fixed_costs]
if missing_specialties:
    print("WARNING: some specialties in 'Patients' are missing from 'Specialties':")
    for s in missing_specialties:
        print(f"   - {s}")
    print("This might cause errors in cost calculations.\n")

# Display extracted costs
print("\n--- Costs per Specialty ---")
for s in specialties:
    if s in base_fixed_costs:
        print(f"{s}: Fixed Cost = {base_fixed_costs[s]} €, Overtime Cost = {base_overtime_costs[s]} €/h")
    else:
        print(f"{s}: Missing data in 'Specialties' sheet")


--- Costs per Specialty ---
ORTHO: Fixed Cost = 2300 €, Overtime Cost = 750 €/h
GEN: Fixed Cost = 1400 €, Overtime Cost = 375 €/h
ENT: Fixed Cost = 1400 €, Overtime Cost = 375 €/h
OBGYN: Fixed Cost = 2300 €, Overtime Cost = 750 €/h
VASCULAR: Fixed Cost = 4100 €, Overtime Cost = 1500 €/h


**3. Parameter Definition**

We define the planning parameters. For the durations, we use a "prudence factor" ($\alpha$) to account for uncertainty :

$Planned\_Duration = Expected\_Duration + \alpha \times Max\_Additional\_Duration$

In [6]:
# --- 3. Parameters definition ---

# Required specialty for each patient
patient_specialty = {f"P{row.Patient_ID}": row.Specialty for _, row in df.iterrows()}

# Planned duration = expected duration + alpha * max additional duration
durations = {
    f"P{row.Patient_ID}": row.Duration + ALPHA_PRUDENCE * row.Overtime
    for _, row in df.iterrows()
}

# Time constraints
H = 8.0         # Standard daily working hours
H_max_sup = 2.0 # Maximum allowed overtime hours

**4. Daily Adjusted Costs Calculation**

Costs are subject to weekend surcharges: +25% on Saturdays and +80% on Sundays. We pre-calculate $C_{jk}$ (fixed cost) and $C'_{jk}$ (overtime cost) for every specialty $j$ on every day $k$.

In [7]:
# --- 4. Pre-calculation of daily costs C_jk and C'_jk ---
C_jk = {}
C_prime_jk = {}

for j in specialties:
    for k in days:
        multiplier = 1.0
        if k == 6:    # Saturday
            multiplier = 1.25
        elif k == 7:  # Sunday
            multiplier = 1.8

        C_jk[(j, k)] = base_fixed_costs[j] * multiplier
        C_prime_jk[(j, k)] = base_overtime_costs[j] * multiplier

**5. Optimization Model with PuLP**

We initialize a Linear Programming problem to minimize the total cost. We define three types of decision variables:
* $x_{ik}$: Binary variable, 1 if patient $i$ is assigned to day $k$.
* $y_{jk}$: Binary variable, 1 if the operating room is opened for specialty $j$ on day $k$.
* $\tau_{jk}$: Continuous variable, representing the planned overtime hours.

In [8]:
# --- 5. Modeling with PuLP ---
model = pulp.LpProblem("Surgical_Planning", pulp.LpMinimize)

# Decision Variables
x = pulp.LpVariable.dicts("X", (patients, days), cat='Binary')           # 1 if patient i is on day k
y = pulp.LpVariable.dicts("Y", (specialties, days), cat='Binary')        # 1 if specialty j works on day k
tau = pulp.LpVariable.dicts("Tau", (specialties, days), lowBound=0)      # Overtime hours

# Objective Function
fixed_cost_expr = pulp.lpSum(C_jk[(j, k)] * y[j][k] for j in specialties for k in days)
overtime_cost_expr = pulp.lpSum(C_prime_jk[(j, k)] * tau[j][k] for j in specialties for k in days)
model += fixed_cost_expr + overtime_cost_expr, "Total_Cost"

**6. Constraints**

In [9]:
# --- Constraints ---

# Each patient is assigned to exactly one day
for i in patients:
    model += pulp.lpSum(x[i][k] for k in days) == 1, f"Patient_Assignment_{i}"

# Only one specialty per day
for k in days:
    model += pulp.lpSum(y[j][k] for j in specialties) <= 1, f"Unique_Specialty_Day_{k}"

# Consistency: Patient can only be assigned to a day when their specialty is scheduled
for i in patients:
    for k in days: 
        model += x[i][k] <= y[patient_specialty[i]][k], f"Consistency_{i}_{patient_specialty[i]}_{k}"

# Overtime calculation
for j in specialties:
    for k in days:
        total_duration = pulp.lpSum(durations[i] * x[i][k] for i in patients if patient_specialty[i] == j)
        model += total_duration - H * y[j][k] <= tau[j][k], f"Overtime_Calc_{j}_{k}"

# Overtime limitation
for j in specialties:
    for k in days:
        model += tau[j][k] <= H_max_sup * y[j][k], f"Overtime_Limit_{j}_{k}"

**7. Solving and Result Analysis**

*N.B. I used Gurobi, but you can change it to use CPLEX or the regular PuLP solver (CBC) with `pulp.PULP_CBC_CMD()`.*

In [11]:
# --- 7. Results Display ---
if pulp.LpStatus[model.status] == 'Optimal':
    print(f"\nOptimal Total Cost: {pulp.value(model.objective):.2f} €")
    print(f"(Prudence Factor Alpha = {ALPHA_PRUDENCE})")

    # List to store data for our summary table
    schedule_data = []

    print("\n--- Surgical Schedule ---")
    for k in days:
        day_has_surgery = False
        for j in specialties:
            if y[j][k].varValue > 0.5:
                day_has_surgery = True
                patients_on_day = [i for i in patients if x[i][k].varValue > 0.5]
                total_time = sum(durations[p] for p in patients_on_day)
                overtime = tau[j][k].varValue if tau[j][k].varValue > 0.01 else 0.0
                
                # Text output
                print(f"\nDay {k} ({day_map[k]}) - Specialty: {j}")
                print(f"  - Scheduled Patients: {', '.join(patients_on_day)}")
                print(f"  - Planned Time (prudent): {total_time:.2f}h")
                if overtime > 0:
                    print(f"  - Planned Overtime: {overtime:.2f}h")
                
                # Append to table data
                schedule_data.append({
                    "Day": day_map[k],
                    "Specialty": j,
                    "Patients": ", ".join(patients_on_day),
                    "Planned Time (h)": round(total_time, 2),
                    "Overtime (h)": round(overtime, 2)
                })
                
        if not day_has_surgery:
            print(f"\nDay {k} ({day_map[k]}) - No surgeries scheduled")
            schedule_data.append({
                "Day": day_map[k],
                "Specialty": "-",
                "Patients": "-",
                "Planned Time (h)": 0.0,
                "Overtime (h)": 0.0
            })

    # --- Global Summary ---
    total_overtime = sum(tau[j][k].varValue for j in specialties for k in days)
    total_days_used = sum(1 for j in specialties for k in days if y[j][k].varValue > 0.5)
    print("\n--- Global Summary ---")
    print(f"Total operating days used: {total_days_used}")
    print(f"Total planned overtime: {total_overtime:.2f} hours\n")

    # --- Create DataFrame ---
    df_schedule = pd.DataFrame(schedule_data)
    
else:
    print("No optimal solution was found.")


Optimal Total Cost: 17813.75 €
(Prudence Factor Alpha = 0.5)

--- Surgical Schedule ---

Day 1 (Monday) - Specialty: ORTHO
  - Scheduled Patients: P0, P11, P13, P20, P28, P32, P36, P42, P47
  - Planned Time (prudent): 8.54h
  - Planned Overtime: 0.55h

Day 2 (Tuesday) - Specialty: ORTHO
  - Scheduled Patients: P2, P12, P25, P26, P35, P40, P43, P50
  - Planned Time (prudent): 8.37h
  - Planned Overtime: 0.37h

Day 3 (Wednesday) - Specialty: VASCULAR
  - Scheduled Patients: P9, P24, P30, P44, P45
  - Planned Time (prudent): 8.30h
  - Planned Overtime: 0.30h

Day 4 (Thursday) - Specialty: ENT
  - Scheduled Patients: P4, P8, P16, P18, P21, P22, P31, P33, P39, P53
  - Planned Time (prudent): 6.80h

Day 5 (Friday) - Specialty: OBGYN
  - Scheduled Patients: P5, P6, P7, P15, P23, P29, P34, P37, P48, P54
  - Planned Time (prudent): 7.21h

Day 6 (Saturday) - Specialty: ENT
  - Scheduled Patients: P14, P17, P41, P46, P49, P51, P52
  - Planned Time (prudent): 6.95h

Day 7 (Sunday) - Specialty: GE

### Timetable

In [12]:
df_schedule

,Day,Specialty,Patients,Planned Time (h),Overtime (h)
0,Monday,ORTHO,"P0, P11, P13, P20, P28, P32, P36, P42, P47",8.54,0.55
1,Tuesday,ORTHO,"P2, P12, P25, P26, P35, P40, P43, P50",8.37,0.37
2,Wednesday,VASCULAR,"P9, P24, P30, P44, P45",8.30,0.30
3,Thursday,ENT,"P4, P8, P16, P18, P21, P22, P31, P33, P39, P53",6.80,0.00
4,Friday,OBGYN,"P5, P6, P7, P15, P23, P29, P34, P37, P48, P54",7.21,0.00
5,Saturday,ENT,"P14, P17, P41, P46, P49, P51, P52",6.95,0.00
6,Sunday,GEN,"P1, P3, P10, P19, P27, P38, P55",7.76,0.00


### 8. Conclusion

The deterministic model successfully found an optimal surgical schedule with a total cost of **17,813.75 €** (using a prudence factor of $\alpha = 0.5$). 

While the total volume of patients requires the operating room to be open every day of the week, the solver's decisions reveal a brilliant dual-optimization strategy to mitigate the heavy weekend financial penalties (1.25x on Saturday and 1.8x on Sunday):

1. **Strict Avoidance of Weekend Overtime:** The model accepts a controlled amount of overtime during weekdays (e.g., 0.55h on Monday, 0.37h on Tuesday) to group patients efficiently. However, it ensures that weekend schedules stay strictly under the standard 8-hour limit (6.95h on Saturday and 7.76h on Sunday), completely dodging the multiplied overtime rates.
2. **Strategic Specialty Allocation:** The model intelligently assigns the **cheapest specialties** to the weekend. By scheduling ENT (Saturday) and GEN (Sunday) — both of which have the lowest base fixed cost (1,400 €) and lowest overtime rate (375 €/h) — the solver drastically minimizes the absolute financial impact of the weekend multipliers. For comparison, placing VASCULAR (4,100 € base fixed cost) on Sunday would have been financially disastrous.

This perfectly demonstrates the power of linear programming: it doesn't just find a feasible schedule; it exploits the cost structure to minimize overall hospital operational expenses while respecting all capacity constraints.